# Bases de Datos

In [1]:
#Posible idea de proponer un farmaco concreto con los principios activos mencionados, existente en CIMA (reducido a si tienen ATC o no denuevo)
import pandas as pd
DDI = pd.read_csv("DDI_sea.csv")
efectos = pd.read_csv("efectos_adversos.csv")
from textos import *
from itertools import product

# Funciones dentro de interaccion

In [ ]:
def principales(df):
    '''
    Busca la lista de las enzimas por las que se metaboliza un principio activo y sus principales en todo caso de haber.

    Parametros
    ------------------
        df- Solo incluye las filas con el ppio activo concreto que estemos consultando
    Devolución
    -----------------------------
        enz - Lista de enzimas por las que se metaboliza el principio activo
        ppal - Lista de enzimas principales por las que se metaboliza el fármaco (prioridad=1)
    '''
    #df con todas las enzimas por las que se metaboliza (no target)
    df_enzimas = df[df["Tipo"]=="enzyme" ]
    # Lista de enzimas unicas por las que se metaboliza el ppio
    enz = df_enzimas["Gene_name"].unique().tolist()      
    #Lista de enzimas principales (si tiene) el ppio (categorizadas como 1)
    ppal = df_enzimas[df_enzimas["Prioridad"]==1]["Gene_name"].unique().tolist()

    return enz, ppal


# Interaccion

In [ ]:
#LODELSTRIP()LOWER() tambien
#Igual en vez de true/false para los booleanos deberia devolver los codigos atc y los ppios o algo asi VER A LA VUELTA si
#PONER LO DE IMPRIMIR EFECTOS ADVERSOS
def interaccion (ppio_1, ppio_2, DDI, efectos_adversos=None, texto=False):
    '''
    Funcion que determina si existe interacción entre los dos principios activos o no.
    Ademas si texto, describe: cada principio y sus enzimas, 
    
    Parámetros
    -----------------
        ppio_1 - El primer principio activo que se quiere comparar con el segundo
        ppio_2 - El segundo principio activo que se quiere comparar con el primero
        DDI - Dataframe que contiene los nombres, ATC, enzimas, target, acciones y prioridad
        efectos_adversos - Dataframe que contiene los efectos adversos de cada ppio, el ATC y la frecuencia con la que aparecen
        texto - Si es necesaria la impresión de texto o no (para comparar el ATC no lo es), por defecto a False
    Devolucion
    ------------------
        Riesgo Leve. Medio y Alto
    '''
    
    #Comprobamos primero que este en nuestra BBDD
    if (ppio_1 in DDI["Drug_name"].values) and (ppio_2 in DDI["Drug_name"].values):
        df_1 = DDI[DDI["Drug_name"] == ppio_1]
        enz_1, ppal_1 = principales(df_1)
        
        df_2 = DDI[DDI["Drug_name"] == ppio_2]
        enz_2, ppal_2 = principales(df_2)
    else: 
        if (ppio_1 in DDI["Drug_name"].values):
            print(f"{ppio_1} no se ha encontrado en la base de datos, porfavor introduce otro")
            #Dar posibles opciones??
        elif (ppio_2 in DDI["Drug_name"].values):
            print(f"{ppio_2} no se ha encontrado en la base de datos, porfavor introduce otro")
            #Dar posibles opciones?
            
    #Sabiendo que está, si queremos texto o no (descripcion de las enz y los ppios)
    intro_1 = texto_intro(ppio_1, enz_1, ppal_1,texto, True) 
    intro_2 = texto_intro(ppio_2, enz_2, ppal_2,texto)

    #Aqui compruebo si hay interaccion o no
    coincidentes = list(set(ppal_1) & set(ppal_2))
    #Calculamos el riesgo y si se desea texto imprime el nivel de riesgo que tendrán
    riesgo = calcular_riesgo(ppio_1, ppio_2,coincidentes, intro_1, intro_2, texto)
    #Si hay interaccion entre ellas se mostrará en coincidentes, sino, la lista será vacia
    if coincidentes:
        #Sacar las acciones y el texto sera para cada enzima por separado
        if texto:
            for e in coincidentes:
                #La fila q contiene la enzima que se quiere ver
                fila_1 = df_1[df_1["Gene_name"]==e]
                #Separamos por si tiene | (no da error si no lo tiene)
                separado_1 = fila_1["Accion"].str.split(r"\|")
                #Nos quedamos con los distintos
                acciones_1 = set(separado_1.explode().tolist())
    
                #Lo mismo pero con el otro principio consultado
                fila_2 = df_2[df_2["Gene_name"]==e]
                separado_2 = fila_2["Accion"].str.split(r"\|")
                acciones_2 = set(separado_2.explode().tolist())

                #Funcion que contiene el texto
                texto_acciones(ppio_1,acciones_1,ppio_2,acciones_2,e) #PARA CADA ENZIMA
                
            #imprimimos los efectos adversos de cada principio 
            #Por cada codigo ATC?
            #Junto todos los efectos adversos q tengan los diferentes ATC con las 5 primeras letras iguales y saco el top 10 si los hay :)


    return riesgo

# ATC

In [4]:
#Tener en cuenta cuando no hay codigo ATC
#Preguntar Alicia si cambio descripcion de funciones (help)
def opciones_ATC(DDI, efectos_adversos, ATC_1, ATC_2, ppio_1, ppio_2):
    """
    Busqueda en la base de datos de opciones con el código ATC de los principios activos problema

    Parámetos
    -------------------------
        DDI - Dataframe que contiene los nombres, ATC, enzimas, target, acciones y prioridad
        efectos_adversos - Dataframe que contiene los efectos adversos de cada ppio, el ATC y la frecuencia con la que aparecen
        ATC_1 - Lista con los códigos ATC únicos del ppio_1
        ATC_2 - Lista con los códigos ATC únicos del ppio_2
        ppio_1 - El primer principio activo que se quiere comparar con el segundo
        ppio_2 - El segundo principio activo que se quiere comparar con el primero
        
    Devolución
    ---------------------------
        #Deberia ser un diccionario HAY QUE CAMBIARLO
        Lista de tuplas con los dos principios activos de alternativa a cada uno que estan categorizados como interaccion leve
    """
    opciones = []
    #Alternativas 1 y 2
    alternativas_1 = []
    alternativas_2 = []
    
    #Primer ppio
    if not ATC_1:
        #Poner opcion de introducir codigo ATC?
        print(f"No ha sido posible buscar alternativas del principio activo: {ppio_1} debido a que no hay datos de cual es su código ATC")
    else:
        for ATC in ATC_1:
            comprobante = False
            #Cogemos los 5 primeras letras del ATC
            i=5
            while comprobante==False :
                #Si hemos llegado a i=0 es que no hay mas codigos atc que comprobar
                if i!=0:
                    #En principio uso las 5 primeras letras del codigo ATC
                    codigo_referencia = ATC[:i]
                    #Obtengo los nombres de los principios que sirven como alternativa
                    principios_1 = DDI[DDI['Drug_ATC'].str.startswith(codigo_referencia, na=False)]["Drug_name"].unique().tolist()
                    
                    if principios_1:
                        #Junto las listas para obtener unas con todas las alternativas del primer principio
                        alternativas_1 = alternativas_1 + principios_1
                        #Imprimo las diferentes alternativas
                        print(f"Para el principio: {ppio_1} con código ATC: {ATC} existen las siguientes alternativas:{principios_1} con código de referencia:{codigo_referencia}")
                        #Establezco el comprobante a True
                        comprobante=True
                    
                else:
                    print(f"No ha sido posible encontrar una opción factible para el principio: {ppio_1} con los códigos ATC")
                    break
                    
                #Si no hay principios uso una letra menos de la q estaba usando 
                i-=1
            
    #Segundo ppio
    if not ATC_2:
        #Poner opcion de introducir codigo ATC?
        print(f"No ha sido posible buscar alternativas del principio activo: {ppio_2} debido a que no hay datos de cual es su código ATC")
    else:
        for ATC in ATC_2:
            comprobante = False
            #Cogemos los 5 primeras letras del ATC
            i=5
            while comprobante==False :
                #Si hemos llegado a i=0 es que no hay mas codigos atc que comprobar
                if i!=0:
                    #En principio uso las 5 primeras letras del codigo ATC
                    codigo_referencia = ATC[:i]
                    #Obtengo los nombres de los principios que sirven como alternativa
                    principios_2 = DDI[DDI['Drug_ATC'].str.startswith(codigo_referencia, na=False)]["Drug_name"].unique().tolist()
                    
                    if principios_2:
                        #Junto las listas para obtener unas con todas las alternativas del primer principio
                        alternativas_2 = alternativas_2 + principios_2
                        #Imprimo las diferentes alternativas
                        print(f"Para el principio: {ppio_2} con código ATC: {ATC} existen las siguientes alternativas:{principios_2} con código de referencia:{codigo_referencia}")
                        #Establezco el comprobante a True
                        comprobante=True
                    
                else:
                    print(f"No ha sido posible encontrar una opción factible para el principio: {ppio_2} con los códigos ATC")
                    break
                    
                #Si no hay principios uso una letra menos de la q estaba usando 
                i-=1
                
    #Si hay elementos en al menos uno de los dos, comprobamos la interaccion de las combinaciones
    if alternativas_1 or alternativas_2:
        #Añado los principios activos
        alternativas_1 = list(set(alternativas_1 + [ppio_1]))
        alternativas_2 = list(set(alternativas_2 + [ppio_2]))
        print()
        print(alternativas_1)
        print(alternativas_2)
        #Combinatoria
        #Deberia sacar combinatorias de ATC por separado dependiendo de su ATC concreto?? igual poner en un diccionario SI
        for comb in product(alternativas_1, alternativas_2):
            interac = interaccion(comb[0], comb[1], DDI)
            if interac=="Leve":
                opciones = opciones + [comb]
        print()
        print()
        print(f"Las posibles combinaciones resultantes son:{opciones}")

    else:
        print("No ha habido ninguna combinacion factible")
        #Es posible que aqui pueda volver a llamar a la funcion pero con un número menos de i???

    return opciones

In [5]:
#Pruebas con casos limite
ppio_1 = "Omeprazole"
ppio_2 = "Clopidogrel"
existe = interaccion (ppio_1, ppio_2, DDI, efectos, texto=True)
if existe=="Alta" or existe=="Media":
    df_1 = DDI[DDI["Drug_name"] == ppio_1]   
    ATC_1 = df_1["Drug_ATC"].unique()
    df_2 = DDI[DDI["Drug_name"] == ppio_2]
    ATC_2 = df_2["Drug_ATC"].unique()
    opciones_ATC(DDI, efectos, ATC_1, ATC_2, ppio_1, ppio_2)

En esta web vamos a indicar si existe una interacción LEVE, MEDIA o ALTA basandonos en las enzimas por las que se metabolizan cada uno de los principios activos consultados en la base de datos de DrugBank

Solo se han tenido en cuenta las siguientes enzimas: [CYP2D6, CYP3A4, CYP3A5, CYP2C19, CYP2C9, CYP1A2] que según DrugBank son las 5 por las que se metabolizan mas principios activos. Teniendo en cuenta tambien la CYP3A5 que es como la CYP3A4 (CAMBIO), sin embargo cuando ambas mencionadas previamente se consideraban principales, la CYP3A5 era eliminada debido a  que la mayoria de la pobalcion no expresa esta enzima.


El principio activo: Omeprazole se metaboliza por las enzimas: ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4', 'CYP2C19'] de las cuales, se ha considerado principal: ['CYP2C19'], en las siguientes descripciones se mostrarán las interacciones solo con dicha enzima considerada principal

El principio activo: Clopidogrel se metaboliza por las enzimas: ['CYP2C9', 'CYP1A2', 'CYP3A4'